# Phase 4 — TFT Global Baseline (Colab GPU, Full Capacity)

This notebook retrains the Phase 4 TFT Global baseline **without the CPU/RAM tradeoffs** from the Mac run.  
Results are saved with `_colab` suffix so both runs coexist in `results/`.

| Setting | Mac CPU run | This Colab run |
|---|---|---|
| `hidden_size` | 16 | **64** |
| `hidden_continuous_size` | 8 | **16** |
| `batch_size` | 32 | **128** |
| `max_epochs` | 25 | **50** |
| `learning_rate` | 1e-3 | **3e-4** |
| `num_workers` | 0 | **4** |
| Accelerator | CPU | **GPU (T4/A100)** |

**Before running:** Runtime → Change runtime type → T4 GPU

## Step 1 — Install Dependencies

In [ ]:
!pip install -q pytorch-forecasting pytorch-lightning lightning pmdarima statsmodels

## Step 2 — Mount Google Drive and Upload Project Files

Upload your project to Google Drive first:
1. Go to drive.google.com
2. Upload the `regime-forecasting` folder (or at minimum `data/raw/master.csv`)
3. Run the cell below to mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Adjust this path to where you uploaded the project in Drive
PROJECT_PATH = '/content/drive/MyDrive/regime-forecasting'

# If you only uploaded master.csv, set this instead:
# PROJECT_PATH = '/content/regime-forecasting'
# !mkdir -p /content/regime-forecasting/data/raw /content/regime-forecasting/results/metrics /content/regime-forecasting/results/figures

os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())
print('master.csv exists:', os.path.exists('data/raw/master.csv'))

### Alternative: Upload master.csv directly (if you didn't upload to Drive)

In [ ]:
# Run this cell ONLY if you skipped Drive mounting and want to upload master.csv directly
# from google.colab import files
# import os
# os.makedirs('/content/regime-forecasting/data/raw', exist_ok=True)
# os.makedirs('/content/regime-forecasting/results/metrics', exist_ok=True)
# os.makedirs('/content/regime-forecasting/results/figures', exist_ok=True)
# uploaded = files.upload()  # select master.csv from your Mac
# import shutil
# shutil.move('master.csv', '/content/regime-forecasting/data/raw/master.csv')
# os.chdir('/content/regime-forecasting')
print('Skipping direct upload (using Drive mount instead)')

## Step 3 — Verify GPU and Imports

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — change runtime to GPU!')

import sys
import pickle
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('lightning.pytorch').setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE
import statsmodels.api as sm
from pmdarima import auto_arima
from pathlib import Path

sns.set_theme(style='darkgrid')
print('All imports OK')

## Step 4 — Configuration

In [ ]:
TRAIN_END  = '2017-12-31'
VAL_START  = '2018-01-01'
VAL_END    = '2020-12-31'

RESULTS = Path('results')
FIGURES = Path('results/figures')
METRICS = Path('results/metrics')
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

# Full-capacity config — no hardware compromises
COLAB_CONFIG = {
    'max_encoder_length': 60,
    'max_prediction_length': 1,
    'hidden_size': 64,           # was 16 on Mac
    'attention_head_size': 4,
    'dropout': 0.1,
    'hidden_continuous_size': 16, # was 8 on Mac
    'learning_rate': 3e-4,
    'batch_size': 128,            # was 32 on Mac
    'max_epochs': 50,             # was 25 on Mac
    'patience': 8,                # was 6 on Mac
    'gradient_clip_val': 0.1,
}
print('Config:', COLAB_CONFIG)

## Step 5 — Helper Functions (inline — no src/ dependency)

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mask = y_true != 0
    dir_acc = float(np.mean(np.sign(y_true[mask]) == np.sign(y_pred[mask])))
    return {'mae': mae, 'rmse': rmse, 'dir_acc': dir_acc}


def prepare_tft_dataframe(df):
    out = df.copy().sort_index()
    out['time_idx'] = np.arange(len(out), dtype=np.int64)
    out['group_id'] = 'global'
    out['fed_funds_lag1']   = out['fed_funds_rate'].shift(1)
    out['yield_spread_lag1'] = out['yield_spread'].shift(1)
    dow   = out.index.dayofweek
    month = out.index.month
    out['dow_sin']   = np.sin(2 * np.pi * dow / 5)
    out['dow_cos']   = np.cos(2 * np.pi * dow / 5)
    out['month_sin'] = np.sin(2 * np.pi * month / 12)
    out['month_cos'] = np.cos(2 * np.pi * month / 12)
    return out.dropna()


def build_tft_dataset(df, config, training_cutoff=None):
    if training_cutoff is None:
        training_cutoff = df['time_idx'].max()
    return TimeSeriesDataSet(
        df[df['time_idx'] <= training_cutoff],
        time_idx='time_idx',
        target='sp500_return',
        group_ids=['group_id'],
        min_encoder_length=config['max_encoder_length'],
        max_encoder_length=config['max_encoder_length'],
        min_prediction_length=config['max_prediction_length'],
        max_prediction_length=config['max_prediction_length'],
        static_categoricals=['group_id'],
        time_varying_known_reals=[
            'time_idx', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
            'fed_funds_lag1', 'yield_spread_lag1',
        ],
        time_varying_unknown_reals=['sp500_return', 'vix'],
        target_normalizer=GroupNormalizer(groups=['group_id'], transformation=None),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=False,
    )


def build_global_tft(training_dataset, config):
    return TemporalFusionTransformer.from_dataset(
        training_dataset,
        learning_rate=config['learning_rate'],
        hidden_size=config['hidden_size'],
        attention_head_size=config['attention_head_size'],
        dropout=config['dropout'],
        hidden_continuous_size=config['hidden_continuous_size'],
        loss=MAE(),
        log_interval=0,
        optimizer='adam',
        reduce_on_plateau_patience=4,
    )

print('Helper functions defined')

## Step 6 — Load Data

In [ ]:
df = pd.read_csv('data/raw/master.csv', index_col=0, parse_dates=True).sort_index()

# Drop test set immediately — sacred until Phase 6
df_phase4 = df.loc[:VAL_END].copy()

train_mask = df_phase4.index <= TRAIN_END
val_mask   = df_phase4.index >  TRAIN_END
n_train, n_val = train_mask.sum(), val_mask.sum()

y_val_actual = df_phase4.loc[val_mask, 'sp500_return'].values
val_dates    = df_phase4.index[val_mask]

print(f'Total rows loaded: {len(df_phase4):,}')
print(f'Train: {n_train:,}  ({n_train/len(df_phase4)*100:.1f}%)')
print(f'Val:   {n_val:,}  ({n_val/len(df_phase4)*100:.1f}%)')
print(f'Test set LOCKED — not loaded')

## Step 7 — ARIMA Baseline (identical to Mac run — no tradeoffs here)

In [ ]:
train_returns = df_phase4.loc[train_mask, 'sp500_return'].values
all_returns   = df_phase4['sp500_return'].values

print('Fitting auto_arima on train returns...')
arima = auto_arima(
    train_returns,
    seasonal=False,
    information_criterion='bic',
    stepwise=True,
    suppress_warnings=True,
    max_p=5, max_q=5, max_d=2,
    error_action='ignore',
)
print(f'Selected order: {arima.order}  |  BIC: {arima.bic():.1f}')

# Apply frozen parameters via Kalman filter — proper 1-step-ahead
arima_full = sm.tsa.SARIMAX(all_returns, order=arima.order).filter(arima.params())
val_start_idx = int(np.where(val_mask)[0][0])
val_end_idx   = int(np.where(val_mask)[0][-1])
pred = arima_full.get_prediction(start=val_start_idx, end=val_end_idx, dynamic=False)
y_val_arima = pred.predicted_mean

metrics_arima = compute_metrics(y_val_actual, y_val_arima)
print(f'ARIMA — MAE={metrics_arima["mae"]:.5f}  RMSE={metrics_arima["rmse"]:.5f}  DirAcc={metrics_arima["dir_acc"]*100:.2f}%')

## Step 8 — Build TFT Datasets

In [ ]:
torch.set_float32_matmul_precision('high')
pl.seed_everything(42, workers=True)

tft_df = prepare_tft_dataframe(df_phase4)
training_cutoff = tft_df.loc[tft_df.index <= TRAIN_END, 'time_idx'].max()
print(f'Training cutoff (time_idx): {training_cutoff}')

training_ds = build_tft_dataset(tft_df, COLAB_CONFIG, training_cutoff=training_cutoff)
validation_ds = TimeSeriesDataSet.from_dataset(
    training_ds, tft_df,
    predict=False,
    stop_randomization=True,
    min_prediction_idx=training_cutoff + 1,
)

train_dl = training_ds.to_dataloader(
    train=True, batch_size=COLAB_CONFIG['batch_size'], num_workers=4, persistent_workers=True
)
val_dl = validation_ds.to_dataloader(
    train=False, batch_size=256, num_workers=4, persistent_workers=True
)
print(f'Train sequences: {len(training_ds):,}  |  Val sequences: {len(validation_ds):,}')

## Step 9 — Build and Train TFT (GPU)

In [ ]:
model = build_global_tft(training_ds, COLAB_CONFIG)
n_params = sum(p.numel() for p in model.parameters())
print(f'TFT parameters: {n_params:,}  (Mac run had ~10,000; this has ~{n_params:,})')

early_stop = EarlyStopping(
    monitor='val_loss', patience=COLAB_CONFIG['patience'], mode='min', verbose=True
)
checkpoint_cb = ModelCheckpoint(
    dirpath='results',
    filename='tft_global_colab_best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
)

trainer = pl.Trainer(
    max_epochs=COLAB_CONFIG['max_epochs'],
    accelerator='gpu',
    devices=1,
    gradient_clip_val=COLAB_CONFIG['gradient_clip_val'],
    callbacks=[early_stop, checkpoint_cb],
    enable_progress_bar=True,
    enable_model_summary=True,
    logger=False,
    num_sanity_val_steps=0,
)

print('\nTraining on GPU...')
trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)
stopped_epoch = trainer.current_epoch
print(f'\nDone. Stopped at epoch {stopped_epoch}')

## Step 10 — Validate and Save

In [ ]:
print('Generating val predictions...')
raw = model.predict(
    val_dl, mode='prediction', return_x=False,
    trainer_kwargs={'accelerator': 'gpu', 'devices': 1, 'logger': False, 'enable_progress_bar': False}
)
y_val_tft_colab = raw.cpu().numpy().flatten()

n_pred = len(y_val_tft_colab)
y_actual_aligned = y_val_actual[-n_pred:] if n_pred < len(y_val_actual) else y_val_actual
val_dates_aligned = val_dates[-n_pred:] if n_pred < len(val_dates) else val_dates

metrics_tft_colab = compute_metrics(y_actual_aligned, y_val_tft_colab)
print(
    f'TFT Global (Colab): '
    f'MAE={metrics_tft_colab["mae"]:.5f}  '
    f'RMSE={metrics_tft_colab["rmse"]:.5f}  '
    f'DirAcc={metrics_tft_colab["dir_acc"]*100:.2f}%'
)

# Save model state dict
torch.save(model.state_dict(), 'results/tft_global_colab.ckpt')

# Save predictions pickle
with open('results/tft_global_colab_predictions.pkl', 'wb') as f:
    pickle.dump({
        'val_dates':     val_dates_aligned,
        'y_actual':      y_actual_aligned,
        'y_pred':        y_val_tft_colab,
        'metrics':       metrics_tft_colab,
        'config':        COLAB_CONFIG,
        'stopped_epoch': stopped_epoch,
    }, f)

print('Artifacts saved.')

## Step 11 — Gate Checks

In [ ]:
for m, name in [(metrics_arima, 'ARIMA'), (metrics_tft_colab, 'TFT Colab')]:
    assert 0.40 < m['dir_acc'] < 0.60, f'FAIL {name}: DirAcc={m["dir_acc"]:.3f} — check for lookahead bias'
    assert 0.001 < m['mae'] < 0.05,    f'FAIL {name}: MAE={m["mae"]:.5f} outside plausible range'

print('All gate checks PASSED')
print()
print(f'ARIMA           MAE={metrics_arima["mae"]:.5f}  RMSE={metrics_arima["rmse"]:.5f}  DirAcc={metrics_arima["dir_acc"]*100:.2f}%')
print(f'TFT Colab (GPU) MAE={metrics_tft_colab["mae"]:.5f}  RMSE={metrics_tft_colab["rmse"]:.5f}  DirAcc={metrics_tft_colab["dir_acc"]*100:.2f}%')

## Step 12 — Compare Colab vs Mac Run

In [ ]:
import os

rows = []

# Colab results (just computed)
rows.append({
    'run': 'TFT Colab GPU',
    'hidden_size': COLAB_CONFIG['hidden_size'],
    'batch_size':  COLAB_CONFIG['batch_size'],
    'epochs':      stopped_epoch,
    'mae':         metrics_tft_colab['mae'],
    'rmse':        metrics_tft_colab['rmse'],
    'dir_acc':     metrics_tft_colab['dir_acc'],
})

# Mac CPU results (if the pkl exists)
mac_pkl = 'results/tft_global_predictions.pkl'
if os.path.exists(mac_pkl):
    with open(mac_pkl, 'rb') as f:
        mac = pickle.load(f)
    rows.append({
        'run': 'TFT Mac CPU',
        'hidden_size': mac['config']['hidden_size'],
        'batch_size':  mac['config']['batch_size'],
        'epochs':      mac['stopped_epoch'],
        'mae':         mac['metrics']['mae'],
        'rmse':        mac['metrics']['rmse'],
        'dir_acc':     mac['metrics']['dir_acc'],
    })
else:
    print('Mac pkl not found — only Colab results shown')

# ARIMA (same in both runs)
rows.append({
    'run': 'ARIMA',
    'hidden_size': '-', 'batch_size': '-', 'epochs': '-',
    'mae':     metrics_arima['mae'],
    'rmse':    metrics_arima['rmse'],
    'dir_acc': metrics_arima['dir_acc'],
})

comparison = pd.DataFrame(rows)
comparison['dir_acc_pct'] = comparison['dir_acc'].apply(
    lambda x: f'{float(x)*100:.2f}%' if x != '-' else '-'
)
print(comparison[['run', 'hidden_size', 'batch_size', 'epochs', 'mae', 'rmse', 'dir_acc_pct']].to_string(index=False))

comparison.to_csv('results/metrics/phase4_tft_comparison.csv', index=False)
print('\nSaved: results/metrics/phase4_tft_comparison.csv')

## Step 13 — Visualise: Colab vs Mac Predictions

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11))
fig.suptitle('Phase 4 — Colab GPU TFT vs Mac CPU TFT vs Actual', fontsize=13)

# Full val period
ax = axes[0]
ax.plot(val_dates_aligned, y_actual_aligned, color='black', linewidth=0.7, label='Actual', alpha=0.85)
ax.plot(val_dates_aligned, y_val_tft_colab, color='darkorange', linewidth=0.7, label='TFT Colab (GPU)', alpha=0.8)
if os.path.exists(mac_pkl):
    with open(mac_pkl, 'rb') as f:
        mac_data = pickle.load(f)
    ax.plot(mac_data['val_dates'], mac_data['y_pred'], color='steelblue', linewidth=0.7,
            label='TFT Mac (CPU)', alpha=0.7, linestyle='--')
ax.axhline(0, color='gray', linewidth=0.4, linestyle='--')
ax.set_title('Validation Period (2018–2020)')
ax.set_ylabel('Daily Return')
ax.legend()

# COVID zoom
ax = axes[1]
covid = (val_dates_aligned >= '2020-02-01') & (val_dates_aligned <= '2020-04-30')
ax.plot(val_dates_aligned[covid], y_actual_aligned[covid], color='black', linewidth=1.0, label='Actual', marker='o', markersize=2)
ax.plot(val_dates_aligned[covid], y_val_tft_colab[covid], color='darkorange', linewidth=1.0, label='TFT Colab', marker='s', markersize=2)
ax.axhline(0, color='gray', linewidth=0.4, linestyle='--')
ax.set_title('Zoom: COVID Crash (Feb–Apr 2020)')
ax.set_ylabel('Daily Return')
ax.legend()

# Residuals
ax = axes[2]
residuals = y_actual_aligned - y_val_tft_colab
colors = ['#4CAF50' if r >= 0 else '#F44336' for r in residuals]
ax.bar(val_dates_aligned, residuals, color=colors, width=1.5, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Residuals (Actual − TFT Colab Predicted)')
ax.set_ylabel('Residual')

plt.tight_layout()
plt.savefig('results/figures/viz_4_colab_tft_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/viz_4_colab_tft_forecast.png')

## Step 14 — Download Results to Your Mac

Run this cell to download the Colab artifacts back to your local machine.

In [ ]:
from google.colab import files

files.download('results/tft_global_colab.ckpt')
files.download('results/tft_global_colab_predictions.pkl')
files.download('results/metrics/phase4_tft_comparison.csv')
files.download('results/figures/viz_4_colab_tft_forecast.png')
print('Downloads triggered — check your browser downloads folder')
print('Move these files into your local regime-forecasting/results/ directory')